# Herbie HRRR probe — KMDW peak-window temperature

**Goal.** Empirically validate the Herbie API surface (which methods exist, what variable names cfgrib hands us, how nearest-grid-cell logic shakes out) **before** committing to a v3 architecture that depends on it. Several of my earlier Herbie API claims (`find_grib()`, `nearest_points()`, source-priority order) are unverified from the docs page I fetched — this notebook resolves them by direct inspection.

**Pipeline being probed.**
1. Pull one historical HRRR run that was operationally available at a known prediction time.
2. Subset to 2 m temperature via `H.xarray("TMP:2 m")` (verified docs syntax).
3. Locate the nearest grid cell to KMDW manually (projection-agnostic, no dependency on unverified helpers).
4. Loop through the noon–4 PM CDT peak window.
5. Compute candidate aggregate features and sanity-check vs. the known actual CLI high.

**Target run.** 2024-07-15 06Z (HRRR). This is the latest HRRR run guaranteed published before a T = 6 AM CDT (11 UTC) prediction — init at 1 AM CDT, fully on AWS/NOMADS by ~2 AM CDT. Lead hours 11–15 cover 17–21 UTC = noon–4 PM CDT, the heating peak.

**Known ground truth.** Actual CLI high for 2024-07-15 = **91°F** (per `predict_summary` smoke test in `model.ipynb`).

## §0 — Install

The `cfgrib` + `eccodes` stack that Herbie depends on is finicky on pure pip (especially on Windows). Use conda-forge:

```
conda install -c conda-forge herbie-data
```

That pulls eccodes binaries that work cleanly on Windows. Pure `pip install herbie-data` may fail to install eccodes correctly. If the import in the next cell fails, run the conda command above and restart the kernel.

In [1]:
import numpy as np
from herbie import Herbie

# ---- constants ----
KMDW_LAT = 41.7868
KMDW_LON = -87.7522

TARGET_DATE     = "2024-07-15"
ACTUAL_CLI_HIGH = 91   # °F, from existing notebook smoke test

# For a T = 6 AM CDT (11Z) prediction, the safely-available HRRR run is 06Z
# (init at 1 AM CDT, fully published by ~2 AM CDT -- comfortably before T).
INIT_RUN_UTC = f"{TARGET_DATE} 06:00"

# Peak heating window for KMDW: noon-4 PM CDT = 17-21 UTC. From a 06Z init:
#   fxx = 11 -> 17Z = 12 noon CDT
#   fxx = 15 -> 21Z =  4 PM CDT
LEAD_HOURS = list(range(11, 16))

print(f"Init run:    {INIT_RUN_UTC}Z  (HRRR)")
print(f"Target date: {TARGET_DATE}  (actual cli_high = {ACTUAL_CLI_HIGH}°F)")
print(f"Lead hours:  {LEAD_HOURS}  (valid 17-21Z = noon-4 PM CDT)")

 ╭─▌▌Herbie─────────────────────────────────────────────╮
 │ INFO: Created a default config file.                 │
 │ You may view/edit Herbie's configuration here:       │
 │     C:\Users\kunpa\.config\herbie\config.toml        │
 ╰──────────────────────────────────────────────────────╯

Init run:    2024-07-15 06:00Z  (HRRR)
Target date: 2024-07-15  (actual cli_high = 91°F)
Lead hours:  [11, 12, 13, 14, 15]  (valid 17-21Z = noon-4 PM CDT)


## §1 — API inspection

Instantiate a single Herbie object and dump its public attribute / method surface. This is the part that resolves the *"is `SOURCES` / `find_grib` / `grib_source` real?"* question by direct inspection rather than docs guesswork.

In [2]:
H = Herbie(INIT_RUN_UTC, model="hrrr", product="sfc", fxx=12)

public_attrs = [m for m in dir(H) if not m.startswith("_")]
print(f"Public attributes on H ({len(public_attrs)} total):")
for chunk in [public_attrs[i:i+6] for i in range(0, len(public_attrs), 6)]:
    print(f"  {chunk}")

print()
for attr_name in ("SOURCES", "grib", "grib_source", "idx_source", "model", "product", "date", "fxx"):
    if hasattr(H, attr_name):
        val = getattr(H, attr_name)
        s = repr(val)
        print(f"H.{attr_name} = {s[:140]}")

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2024-Jul-15 06:00 UTC F12 ┊ GRIB2 @ aws ┊ IDX @ aws
Public attributes on H (37 total):
  ['DESCRIPTION', 'DETAILS', 'EXPECT_IDX_FILE', 'IDX_STYLE', 'IDX_SUFFIX', 'LOCALFILE']
  ['PRODUCTS', 'SOURCES', 'config', 'date', 'download', 'find_grib']
  ['find_idx', 'fxx', 'get_localFileName', 'get_localFilePath', 'get_localIndexFilePath', 'get_remoteFileName']
  ['grib', 'grib_source', 'help', 'idx', 'idx_source', 'index_as_dataframe']
  ['inventory', 'model', 'overwrite', 'priority', 'product', 'product_description']
  ['save_dir', 'search_help', 'tell_me_everything', 'terrain', 'valid_date', 'verbose']
  ['xarray']

H.SOURCES = {'aws': 'https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrrr.20240715/conus/hrrr.t06z.wrfsfcf12.grib2', 'nomads': 'https://nomads.ncep.noaa.gov/
H.grib = 'https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrrr.20240715/conus/hrrr.t06z.wrfsfcf12.grib2'
H.grib_source = 'aws'
H.idx_source = 'aws'
H.model = 'hrrr'
H.product = 'sfc'
H.date = Time

## §2 — Pull 2 m temperature subset

`H.xarray("TMP:2 m")` is the verified-from-docs syntax for subsetting a single GRIB2 record by name. This avoids downloading the full HRRR file (~1.5 GB) — only the requested record is fetched and parsed.

Inspect the returned `xarray.Dataset`: which variable name did cfgrib assign? What coordinates are exposed? What are the grid dimensions?

In [3]:
ds = H.xarray("TMP:2 m")
print(ds)
print()
print(f"data_vars: {list(ds.data_vars.keys())}")
print(f"coords:    {list(ds.coords.keys())}")
print(f"dims:      {dict(ds.sizes)}")

<xarray.Dataset> Size: 38MB
Dimensions:              (y: 1059, x: 1799)
Coordinates:
    latitude             (y, x) float64 15MB 21.14 21.15 21.15 ... 47.85 47.84
    longitude            (y, x) float64 15MB 237.3 237.3 237.3 ... 299.0 299.1
    time                 datetime64[ns] 8B 2024-07-15T06:00:00
    step                 timedelta64[ns] 8B 12:00:00
    heightAboveGround    float64 8B 2.0
    valid_time           datetime64[ns] 8B 2024-07-15T18:00:00
    gribfile_projection  object 8B None
Dimensions without coordinates: y, x
Data variables:
    t2m                  (y, x) float32 8MB 297.1 297.1 297.1 ... 290.0 289.8
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    model:                   hrrr
    product:                 sfc
    description:            

## §3 — Nearest grid cell to KMDW

HRRR is on a Lambert Conformal projection, not regular lat/lon — so we can't use `ds.sel(lat=..., lon=..., method="nearest")` directly. The robust approach is to compute squared distance over the full 2D lat/lon arrays and take `argmin`. Works regardless of projection and doesn't depend on the unverified `H.herbie.nearest_points()` helper.

If the longitude grid uses the 0–360 convention, normalize to −180..180 first.

In [4]:
temp_var = list(ds.data_vars.keys())[0]   # whatever cfgrib named the 2m-temp field
lat_arr = ds.latitude.values
lon_arr = ds.longitude.values
if (lon_arr > 180).any():
    lon_arr = np.where(lon_arr > 180, lon_arr - 360, lon_arr)

dist_sq = (lat_arr - KMDW_LAT) ** 2 + (lon_arr - KMDW_LON) ** 2
iy, ix = np.unravel_index(dist_sq.argmin(), dist_sq.shape)

print(f"Variable used:       {temp_var}")
print(f"Nearest grid cell:   ({lat_arr[iy, ix]:.4f}°N, {lon_arr[iy, ix]:.4f}°W)")
print(f"Distance from KMDW:  ~{np.sqrt(dist_sq[iy, ix]) * 111:.2f} km")
print(f"Index (iy, ix):      ({iy}, {ix})  [reuse for all lead hours]")

# Sanity check the K -> F conversion
t_k_check = float(ds[temp_var].isel(y=iy, x=ix).values)
t_f_check = (t_k_check - 273.15) * 9 / 5 + 32
print(f"\nfxx=12 (valid 18Z = 1 PM CDT)  ->  {t_k_check:.2f} K = {t_f_check:.2f}°F")

Variable used:       t2m
Nearest grid cell:   (41.7781°N, -87.7363°W)
Distance from KMDW:  ~2.02 km
Index (iy, ix):      (665, 1169)  [reuse for all lead hours]

fxx=12 (valid 18Z = 1 PM CDT)  ->  302.45 K = 84.75°F


## §4 — Pull all peak-window lead hours

Reuse the `(iy, ix)` indices computed above (the HRRR grid doesn't change between lead hours of the same run) and iterate fxx = 11..15. Each call hits the AWS / NOMADS cache and pulls just the `TMP:2 m` record — should take a few seconds per hour.

In [5]:
peak_temps_f = []
for fxx in LEAD_HOURS:
    H_fxx = Herbie(INIT_RUN_UTC, model="hrrr", product="sfc", fxx=fxx)
    ds_fxx = H_fxx.xarray("TMP:2 m")
    t_k = float(ds_fxx[temp_var].isel(y=iy, x=ix).values)
    t_f = (t_k - 273.15) * 9 / 5 + 32
    valid_utc = (6 + fxx) % 24
    valid_cdt = (valid_utc - 5) % 24
    print(f"fxx={fxx:>2}  valid {valid_utc:02d}Z = {valid_cdt:>2}:00 CDT:  T_2m = {t_f:.2f}°F")
    peak_temps_f.append(t_f)

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2024-Jul-15 06:00 UTC F11 ┊ GRIB2 @ aws ┊ IDX @ aws
fxx=11  valid 17Z = 12:00 CDT:  T_2m = 82.08°F
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2024-Jul-15 06:00 UTC F12 ┊ GRIB2 @ aws ┊ IDX @ local
fxx=12  valid 18Z = 13:00 CDT:  T_2m = 84.75°F
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2024-Jul-15 06:00 UTC F13 ┊ GRIB2 @ aws ┊ IDX @ aws
fxx=13  valid 19Z = 14:00 CDT:  T_2m = 86.37°F
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2024-Jul-15 06:00 UTC F14 ┊ GRIB2 @ aws ┊ IDX @ aws
fxx=14  valid 20Z = 15:00 CDT:  T_2m = 87.89°F
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2024-Jul-15 06:00 UTC F15 ┊ GRIB2 @ aws ┊ IDX @ aws
fxx=15  valid 21Z = 16:00 CDT:  T_2m = 89.92°F


## §5 — Candidate aggregate features + sanity check

Four candidate features off the peak window: `max`, `mean`, `min`, `p90`. The peak-max is the strongest single feature for predicting CLI high (which is itself a daily max over a similar local-time window).

If `|HRRR peak-max − actual CLI high|` is **< ~3°F**, the pipeline produces physically sensible KMDW forecasts and we can wire it into `feature_df`. If **> ~5°F**, recheck the grid-cell lookup and the K → °F conversion before trusting these as features.

In [6]:
hrrr_temp_max_peak  = float(max(peak_temps_f))
hrrr_temp_mean_peak = float(np.mean(peak_temps_f))
hrrr_temp_min_peak  = float(min(peak_temps_f))
hrrr_temp_p90_peak  = float(np.percentile(peak_temps_f, 90))

print(f"Candidate HRRR peak-window features ({TARGET_DATE}):")
print(f"  hrrr_temp_max_peak  = {hrrr_temp_max_peak:.2f}°F  <- strongest signal for cli_high")
print(f"  hrrr_temp_mean_peak = {hrrr_temp_mean_peak:.2f}°F")
print(f"  hrrr_temp_min_peak  = {hrrr_temp_min_peak:.2f}°F")
print(f"  hrrr_temp_p90_peak  = {hrrr_temp_p90_peak:.2f}°F")
print()
print(f"Actual cli_high on {TARGET_DATE}: {ACTUAL_CLI_HIGH}°F")
print(f"HRRR peak-max forecast:  {hrrr_temp_max_peak:.2f}°F")
print(f"|delta|:                 {abs(ACTUAL_CLI_HIGH - hrrr_temp_max_peak):.2f}°F")

Candidate HRRR peak-window features (2024-07-15):
  hrrr_temp_max_peak  = 89.92°F  <- strongest signal for cli_high
  hrrr_temp_mean_peak = 86.20°F
  hrrr_temp_min_peak  = 82.08°F
  hrrr_temp_p90_peak  = 89.11°F

Actual cli_high on 2024-07-15: 91°F
HRRR peak-max forecast:  89.92°F
|delta|:                 1.08°F


## §6 — Next steps once §1–§5 run clean

Assuming the sanity check passes:

1. **Wrap §3–§5 into a reusable function** `get_hrrr_peak_features(date, init_hour_utc=6)` returning a dict of the four aggregate features.
2. **Backfill** features over the training range (2020-01-01 → present) using the Historical Forecast archive (HRRR available back to 2018-01-01).
3. **Join** the new columns into `feature_df` in §3.x of `model_para_v2.ipynb` (our strongest current artifact) and re-run §5 → §10.2 to measure the CRPS lift from adding HRRR forecast features.
4. **Init-hour discipline.** For T = 6 AM CDT (v1) use the 06Z HRRR run; for T = 1 PM CDT (v2) use the 12Z run (init at 7 AM CDT, fully published by ~8 AM CDT). Hard-code these to prevent train/serve skew.

## §7 — Reusable raw fetch (multi-variable, full forecast horizon)

Same philosophy as the IEM fetchers (`fetch_metar`, `fetch_taf`, `fetch_cli`, `fetch_asos_1min`): **dump the full raw archive, aggregate later in feature engineering**. No per-day pre-aggregation, no peak-window pre-filtering — those decisions belong in the model notebook's §3, not here.

**What we fetch per `(date, init_hour_utc)`:**

| Field name | GRIB record | Units | Counterpart in METAR/TAF |
|---|---|---|---|
| `t2m`   | TMP:2 m above ground   | K   | METAR `temp_c`     |
| `d2m`   | DPT:2 m above ground   | K   | METAR `dewp_c`     |
| `u10`   | UGRD:10 m above ground | m/s | METAR `wspd_kt` (combined w/ v10) |
| `v10`   | VGRD:10 m above ground | m/s | METAR `wdir`     (combined w/ u10) |
| `gust`  | GUST:surface           | m/s | METAR `gust_kt`    |
| `prmsl` | PRMSL:mean sea level   | Pa  | METAR `slp_mb`     |
| `tcc`   | TCDC:entire atmosphere | %   | METAR sky-cover frac |
| `cape`  | CAPE:surface           | J/kg | (no METAR analogue — net new feature) |
| `lftx`  | LFTX:500-1000 mb       | K   | (no METAR analogue) |

All native units preserved in the parquet — let downstream feature engineering convert K → °F, m/s → kt, Pa → mb to match the model's existing column conventions.

**Forecast-hour coverage.** Synoptic runs (00Z, 06Z, 12Z, 18Z) go out to **+48 h**; hourly runs go to **+18 h**. The function auto-caps `fxx_max` based on `init_hour_utc` to avoid 404s on lead hours past horizon.

**Disk hygiene.** Same as IEM fetchers — Herbie caches the subset GRIB locally, we extract, we delete. Backfill across 2020–2026 stays bounded at a few MB of transient cache.

In [ ]:
import pandas as pd
import xarray as xr
from pathlib import Path

# Combined GRIB search for all variables we want. Each is anchored with leading ":"
# so we don't false-positive match e.g. "TMP:850 mb" when we wanted "TMP:2 m".
HRRR_SEARCH = (
    "(:TMP:2 m above ground"
    "|:DPT:2 m above ground"
    "|:UGRD:10 m above ground"
    "|:VGRD:10 m above ground"
    "|:GUST:surface"
    "|:PRMSL:mean sea level"
    "|:TCDC:entire atmosphere"
    "|:CAPE:surface"
    "|:LFTX:500-1000 mb)"
)

HRRR_VARS = ["t2m", "d2m", "u10", "v10", "gust", "prmsl", "tcc", "cape", "lftx"]


def _synoptic_horizon(init_hour_utc):
    """HRRR forecast horizon: 48h for synoptic 00/06/12/18Z runs, else 18h."""
    return 47 if init_hour_utc in (0, 6, 12, 18) else 18


def fetch_hrrr_kmdw_day(date, init_hour_utc=6, fxx_max=None,
                       grid_iy=iy, grid_ix=ix, cleanup=True, verbose=False):
    """Fetch one HRRR run's full forecast horizon for KMDW, all canonical variables.

    Mirrors the IEM fetcher philosophy: pull raw, derive later. Returns one row
    per forecast hour with `len(HRRR_VARS)` variable columns in native units.

    Args:
    - date (str | pd.Timestamp): target date. The HRRR run is at that date's
      `init_hour_utc:00 UTC`. For T=6 AM CDT predictions use init_hour_utc=6
      (06Z run, init at 1 AM CDT, fully published before T).
    - init_hour_utc (int): 0-23. Defaults to 6.
    - fxx_max (int | None): inclusive upper bound on forecast hour. Defaults
      to the model horizon (48 for synoptic, 18 for hourly).
    - grid_iy, grid_ix (int): cached nearest-grid-cell indices from §3.
    - cleanup (bool): if True, unlink the cached GRIB and .idx files after extraction.
    - verbose (bool): if False, suppress Herbie's per-fetch banner.

    Returns:
    - list[dict]: one row per fxx with keys date, init_hour_utc, fxx,
      valid_utc, valid_local + the HRRR_VARS columns.
    """
    date = pd.Timestamp(date).normalize()
    init_run = f"{date.date()} {init_hour_utc:02d}:00"
    if fxx_max is None:
        fxx_max = _synoptic_horizon(init_hour_utc)

    rows = []
    for fxx in range(fxx_max + 1):
        H = Herbie(init_run, model="hrrr", product="sfc", fxx=fxx, verbose=verbose)
        try:
            result = H.xarray(HRRR_SEARCH)
        except Exception as e:
            print(f"  [{date.date()} {init_hour_utc:02d}Z fxx={fxx}] FAILED: {e}")
            continue

        # H.xarray with a multi-record regex can return either a single Dataset
        # (when all matched records share dimensions) or a list of Datasets
        # (when records span multiple vertical level types). Normalize to a list.
        datasets = [result] if isinstance(result, xr.Dataset) else list(result)

        valid_utc_dt = pd.Timestamp(date) + pd.Timedelta(hours=init_hour_utc + fxx)
        valid_local_dt = valid_utc_dt.tz_localize("UTC").tz_convert("America/Chicago")

        row = {
            "date":           date,
            "init_hour_utc":  int(init_hour_utc),
            "fxx":            int(fxx),
            "valid_utc":      valid_utc_dt,
            "valid_local":    valid_local_dt.tz_localize(None),  # naive for parquet
        }
        # Pull each variable from whichever Dataset has it
        for v in HRRR_VARS:
            row[v] = None
        for ds_one in datasets:
            for v_name, da in ds_one.data_vars.items():
                if v_name in HRRR_VARS:
                    try:
                        row[v_name] = float(da.isel(y=grid_iy, x=grid_ix).values)
                    except Exception:
                        pass
        rows.append(row)

        # Disk hygiene -- delete the cached subset GRIB and its .idx
        if cleanup:
            for search_str in HRRR_SEARCH.strip("()").split("|"):
                try:
                    local_path = Path(H.get_localFilePath(search_str.lstrip(":")))
                    if local_path.exists():
                        local_path.unlink()
                    idx_path = local_path.with_suffix(local_path.suffix + ".idx")
                    if idx_path.exists():
                        idx_path.unlink()
                except Exception:
                    pass

    return rows


# Smoke test on the date we already validated
test_rows = fetch_hrrr_kmdw_day("2024-07-15", init_hour_utc=6, fxx_max=5, verbose=False)
test_df = pd.DataFrame(test_rows)
print(f"Returned {len(test_df)} rows × {len(test_df.columns)} cols")
print(f"Columns: {list(test_df.columns)}")
test_df.head()

## §8 — Backfill across the training range → parquet

Same IEM-fetcher pattern: resumable, idempotent, incremental. Writes one parquet per `(init_hour_utc)` since different model anchors (v1 = 06Z, v2 = 12Z) want different parquets.

**Estimated runtime.** ~2,200 days × 48 fxx ≈ **106,000 fetches** at ~1 s each = **~30 hours** for the full 06Z archive. Plan accordingly:
- Split by year: backfill 2020, then 2021, etc.
- Or run overnight on a stable connection.
- Or accept fxx_max=18 (matches hourly-run horizon) which cuts runtime to ~10 hours.

**Resumability.** If the parquet exists, only dates not already covered get fetched. Re-running picks up where it left off.

**File layout.**
- `data/kmdw_hrrr_06z.parquet` — for T=6 AM CDT model (v1)
- `data/kmdw_hrrr_12z.parquet` — for T=1 PM CDT model (v2)

In [ ]:
import time

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def hrrr_parquet_path(init_hour_utc):
    return DATA_DIR / f"kmdw_hrrr_{init_hour_utc:02d}z.parquet"


def build_kmdw_hrrr_archive(start_date, end_date, init_hour_utc=6,
                            fxx_max=None, log_every=10):
    """Incrementally backfill the HRRR raw archive for KMDW.

    Args:
    - start_date, end_date (str): inclusive date range, e.g. "2020-01-01", "2026-05-19".
    - init_hour_utc (int): HRRR init hour to anchor on (6 for v1, 12 for v2).
    - fxx_max (int | None): forecast horizon override. None -> auto (48 for synoptic).
    - log_every (int): progress logging frequency in days.

    Returns:
    - pd.DataFrame of the full archive.
    """
    parquet_path = hrrr_parquet_path(init_hour_utc)
    all_dates = pd.date_range(start_date, end_date, freq="D").normalize()

    if parquet_path.exists():
        existing = pd.read_parquet(parquet_path)
        done_dates = set(pd.to_datetime(existing["date"]).dt.normalize().unique())
        todo_dates = [d for d in all_dates if d not in done_dates]
        print(f"Resuming {parquet_path.name}: {len(done_dates)} dates done, "
              f"{len(todo_dates)} to fetch.")
    else:
        existing = pd.DataFrame()
        todo_dates = list(all_dates)
        print(f"Fresh build of {parquet_path.name}: {len(todo_dates)} dates to fetch.")

    if not todo_dates:
        print("Nothing to do.")
        return existing

    new_rows = []
    t0 = time.time()
    for i, d in enumerate(todo_dates):
        try:
            day_rows = fetch_hrrr_kmdw_day(d, init_hour_utc=init_hour_utc,
                                          fxx_max=fxx_max, verbose=False)
            new_rows.extend(day_rows)
        except Exception as e:
            print(f"  [{d.date()}] FAILED: {e}")
            continue

        if (i + 1) % log_every == 0 or i == len(todo_dates) - 1:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta_min = (len(todo_dates) - i - 1) / rate / 60 if rate > 0 else float("inf")
            print(f"  [{i+1:>4}/{len(todo_dates)}] {d.date()}  "
                  f"elapsed={elapsed/60:5.1f}min  rate={rate:.2f} days/s  "
                  f"eta={eta_min:5.1f}min")

            # Checkpoint: flush to parquet every log_every days so a crash doesn't
            # lose hours of work. Same paranoia as the IEM fetchers' year-at-a-time pull.
            checkpoint_df = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True) \
                if len(existing) else pd.DataFrame(new_rows)
            checkpoint_df.to_parquet(parquet_path)

    full_df = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True) \
        if len(existing) else pd.DataFrame(new_rows)
    full_df = full_df.sort_values(["date", "fxx"]).reset_index(drop=True)
    full_df.to_parquet(parquet_path)
    print(f"\nWrote {len(full_df)} rows ({full_df['date'].nunique()} unique dates) to {parquet_path}")
    return full_df


# Smoke test: 3-day archive, both T anchors. Verifies the loop, parquet roundtrip,
# and resumability before committing to the full backfill.
print("--- Smoke: 06Z anchor, 2024-07-15 to 2024-07-17 ---")
smoke_06 = build_kmdw_hrrr_archive("2024-07-15", "2024-07-17", init_hour_utc=6)
print(f"\n06Z smoke shape: {smoke_06.shape}")
smoke_06.head()

## §9 — Inspect the parquet

Quick eyeball: does the parquet look like what we'd expect for daily-cycle temperature in mid-July at KMDW? A 06Z run forecasting out 48h should show two diurnal cycles, with peaks near 21–22 UTC (4–5 PM CDT).

In [ ]:
one_day = smoke_06[smoke_06["date"] == "2024-07-15"].copy()
one_day["t_F"] = (one_day["t2m"] - 273.15) * 9 / 5 + 32
print(f"06Z run on 2024-07-15: {len(one_day)} forecast hours, peak forecast = {one_day['t_F'].max():.2f}°F")
print(f"Hour of peak forecast (CDT): {one_day.loc[one_day['t_F'].idxmax(), 'valid_local'].hour}:00")
one_day[["fxx", "valid_local", "t2m", "t_F", "d2m", "u10", "v10", "tcc", "cape"]].to_string(index=False)

## §10 — Run the full backfill

Once the smoke test in §8/§9 looks right, kick off the full multi-year backfill. **This is the long one.** Plan for ~30 hours per init-anchor. Safe to interrupt — resumable from parquet.

```python
# v1 model (T=6 AM CDT)
build_kmdw_hrrr_archive("2020-01-01", "2026-05-19", init_hour_utc=6)

# v2 model (T=1 PM CDT)
build_kmdw_hrrr_archive("2020-01-01", "2026-05-19", init_hour_utc=12)
```

After backfill, the raw HRRR archive is on disk as `data/kmdw_hrrr_06z.parquet` and `data/kmdw_hrrr_12z.parquet`. Feature derivation (peak-window aggregates, t/t-1 deltas, run-to-run skill features, etc.) happens in `model_para_v2.ipynb` §3.x — not here. Same separation of concerns as the METAR/TAF/CLI pipeline.